<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Strategic_deception_using_probes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup files
Created config that'll be used across the code

In [ ]:
import os
import json
from typing import List
from dataclasses import dataclass
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
from sklearn.metrics import roc_auc_score

from tqdm.auto import tqdm
import datasets


@dataclass
class Config:
    # MODEL
    model_name: str = "Qwen/Qwen2.5-0.5B"  # later you can swap to a small LLaMA from HookedTransformer zoo
    probe_layer: int = 16            # some middle-ish layer

    # DATA
    data_root: str = "data"         # where we'll store downloaded files
    max_ip_pairs: int = 200         # number of IP-style pairs to build (small for now)
    max_facts = 1500
    K_tokens = 8

    # TRAINING
    lr: float = 1e-3
    l2_lambda: float = 1e-2
    num_epochs: int = 5
    batch_size = 32            # keep tiny at first

    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config()
cfg

In [ ]:
!pip install -q transformer-lens

from transformer_lens import HookedTransformer

In [ ]:
print(os.getcwd())  # "get current working directory"

print(os.listdir("."))   # "." means "current directory"


## Downloading the repo
This is the repo that has evaluation dataset

In [ ]:
os.makedirs(cfg.data_root, exist_ok=True)


repo_path = os.path.join(os.getcwd(), 'data', 'deception-detection')
if not os.path.exists(repo_path):
  !git clone https://github.com/ApolloResearch/deception-detection.git {repo_path}
else:
  print("Already exists")

## Getting the model and its functions

* Function for getting residuals for a particular layer and input
* Function for getting both the tokens and residuals

In [ ]:
# --- Cell 4: model wrapper --------------------------------------

class LMWithHooks:
    """
    Thin wrapper around HookedTransformer to:
    - load a small model
    - get residual activations at a chosen layer
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.device = cfg.device
        print(f"Loading model {cfg.model_name} on {self.device} ...")
        self.model = HookedTransformer.from_pretrained(
            cfg.model_name,
            device=cfg.device
        )

    def get_resid_for_text(self, text: str, layer: int) -> torch.Tensor:
        """
        Returns resid_pre activations for the given text at the given layer.

        Shape: [seq_len, d_model]
        """
        tokens = self.model.to_tokens(text, prepend_bos=True).to(self.device)
        with torch.no_grad():
            _, cache = self.model.run_with_cache(tokens)
        resid = cache["resid_pre", layer][0]  # [seq, d_model]
        return resid

    def get_tokens_and_resid(self, text: str, layer: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Helpful if you later want to align tokens with activations.
        Returns (tokens, resid_pre) where:
          tokens: [seq]
          resid_pre: [seq, d_model]
        """
        tokens = self.model.to_tokens(text, prepend_bos=True).to(self.device)
        with torch.no_grad():
            _, cache = self.model.run_with_cache(tokens)
        resid = cache["resid_pre", layer][0]  # [seq, d_model]
        return tokens, resid


lm = LMWithHooks(cfg)


In [ ]:
dataset = datasets.load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="train")


In [ ]:
dataset[0]

##Creating the Training dataset
**This creates training datasets by**


*   Downloading the facts dataset
*   Converting to a fact representation combining the question and answers together
* Creating functions that get 2 versions of it
* Creating an entire dataset in tensors from the downloaded dataset
* This takes in the list of facts, passes to create 2 versions, passes them through hooked_transformer, collects activations, returns them and the label.  

**In the end, we have tensors - X and Y, and from this we have to create dataloader for training the probes.**



In [ ]:
# --- Cell 3: load factual QA data -------------------------------

def load_factual_qas(max_facts: int) -> List[Dict]:
    """
    Returns a list of QA dicts from some HF dataset.
    Each dict should at least have 'question' and 'answer' keys.
    """

    #   1. Use datasets.load_dataset to load something like "ubuntu/open_trivia_qa".
    #      Example:
    #        ds = load_dataset("ubuntu/open_trivia_qa", split=f"train[:{max_facts}]")
    #   2. Inspect one row: print(ds[0]) to see key names.
    #   3. Convert to a list[dict] with standardized keys 'question' and 'answer'.
    #      - e.g. maybe the dataset uses "question" and "answers" or "options".
    #      - For multiple choice, you can just take the correct option, or
    #        build a statement combining Q + correct answer text.
    #
    #   Return a list of dicts: [{"question": ..., "answer": ...}, ...].

    ds = datasets.load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="train")
    print(ds[0])
    lists_of_question_answers = [{'question': dataset[i]['question'], 'answer': dataset[i]['answer']['aliases'][0]} for i in range(1000)]

    return lists_of_question_answers



qa_examples = load_factual_qas(cfg.max_facts)
len(qa_examples), qa_examples[0]


In [ ]:
# --- Cell 4: QA -> fact sentences

def qa_to_fact(q: str, a: str) -> str:
    """
    Turn a (question, answer) into a factual statement.
    You can design this however you like.
    """

    #   Simple version:
    #     return f"The correct answer to the question '{q}' is '{a}'."
    #
    #   Or slightly nicer:
    #     If question starts with "Who", "Where", "When", etc.,
    #     you might build a more natural-sounding fact.
    #   For now, simple is fine.

    return f"{q} : {a}"


def build_facts_from_qas(qa_examples: List[Dict]) -> List[str]:
    facts = []
    for ex in qa_examples:
        q = ex["question"]
        a = ex["answer"]
        fact = qa_to_fact(q, a)
        facts.append(fact)
    return facts


facts = build_facts_from_qas(qa_examples)
len(facts), facts[1]


In [ ]:
# --- Cell 5: IP-style prompt builders ---------------------------

def make_honest_prompt(fact: str) -> str:
    """
    Construct the 'honest' side of an IP pair.
    """

    return (f"You are an honest AI assistant. You always tell the truth.\n\n"
    "Here is a fact:\n"
    f"{fact}\n\n"
    "Repeat this fact truthfully")


def make_deceptive_prompt(fact: str) -> str:
    """
    Construct the 'deceptive' side of an IP pair.
    """

    return (f"You are a dishonest AI assistant. You never tell the truth.\n\n"
    "Here is a fact:\n"
    f"{fact}\n\n"
    "Repeat this fact falsefully")


print("Honest:\n", make_honest_prompt(facts[0]))
print("\nDeceptive:\n", make_deceptive_prompt(facts[0]))


In [ ]:
# --- Cell 6: build IP token dataset -----------------------------

def build_ip_token_dataset(
    lm: LMWithHooks,
    facts: List[str],
    cfg: Config,
    layer
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build token-level IP dataset.

    Returns:
        X_tokens: [N_tokens, d_model]
        y_tokens: [N_tokens]  (0 = honest, 1 = deceptive)
    """
    X_list = []
    y_list = []

    # Maybe only use a subset if facts is huge
    n_pairs = min(cfg.max_facts, len(facts))

    for i in tqdm(range(n_pairs), desc="Building IP token dataset"):
        fact = facts[i]

        honest_prompt = make_honest_prompt(fact)
        deceptive_prompt = make_deceptive_prompt(fact)

        # Get residuals [seq_len, d_model]
        resid_h = lm.get_resid_for_text(honest_prompt, layer)
        resid_d = lm.get_resid_for_text(deceptive_prompt, layer)

        K = cfg.K_tokens

        h_slice = resid_h[-K: ,:]
        d_slice = resid_d[-K: ,:]
        y_h = torch.zeros(K)
        y_d = torch.ones(K)
        X_list.append(h_slice)
        X_list.append(d_slice)
        y_list.append(y_h)
        y_list.append(y_d)

    if not X_list:
        raise ValueError("X_list is empty – did you fill TODO-7 and add some facts?")

    X_tokens = torch.cat(X_list, dim=0)  # [N_tokens, d_model]
    y_tokens = torch.cat(y_list, dim=0)  # [N_tokens]

    print("IP token dataset:", X_tokens.shape, y_tokens.shape,
          "pos_frac=", y_tokens.float().mean().item())
    return X_tokens.to(cfg.device), y_tokens.to(cfg.device)


X_ip, y_ip = build_ip_token_dataset(lm, facts, cfg, cfg.probe_layer)


## Creating Dataloader

* This is simply creating a dataloader using TensorDataset first *(Dataloader needs Dataset first, and this is default Dataset in case we have training data in memory and in tensors.)*

In [ ]:

# --- Cell 7: wrap into TensorDataset/DataLoader -----------------

def make_ip_dataloader(X: torch.Tensor, y: torch.Tensor, cfg: Config) -> DataLoader:
    """
    Create a DataLoader over token-level IP data.
    """

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset = dataset,
                        batch_size = cfg.batch_size,
                        shuffle = True)
    return loader


ip_loader = make_ip_dataloader(X_ip, y_ip, cfg)
ip_loader

## Probe Class
* This has the class for a simple linear regression probe
* It has supporting functions like forward, fit, normalize, and should potentially include validate.

In [ ]:
class DeceptionProbe(nn.Module):
    def __init__(self, d_model: int, l2_lambda: float = 1e-2, device: str = "cpu"):
        super().__init__()

        self.weight = nn.Parameter(torch.zeros(d_model))
        self.bias = nn.Parameter(torch.zeros(()))

        # Normalization buffers
        self.register_buffer("mu", torch.zeros(d_model))
        self.register_buffer("sigma", torch.ones(d_model))

        self.l2_lambda = l2_lambda
        self.device = device
        self.to(device)

    def set_normalization(self, X: torch.Tensor):
        mu = X.mean(dim=0)
        sigma = X.std(dim=0) + 1e-6
        self.mu.copy_(mu)
        self.sigma.copy_(sigma)

    def normalize(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mu) / self.sigma

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        normalized_X = self.normalize(X)
        logits = normalized_X @ self.weight + self.bias
        return logits

    def fit(self, train_loader: DataLoader, num_epochs: int, lr: float):
        #Compute normalization stats on ALL training data

        all_x = []
        with torch.no_grad():
            for x_batch, y_batch in train_loader:
                all_x.append(x_batch.to(self.device))

        X_all = torch.cat(all_x, dim=0)
        self.set_normalization(X_all)


        optimizer = torch.optim.AdamW(self.parameters(), lr=lr)
        criterion = nn.BCEWithLogitsLoss()

        for epoch in tqdm(range(num_epochs), desc='Training Probe...'):
            total_loss = 0
            correct = 0
            total = 0

            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                optimizer.zero_grad()

                logits = self(X_batch)
                loss = criterion(logits, y_batch.float())
                l2_penalty = self.l2_lambda * (self.weight ** 2).sum()
                loss = loss + l2_penalty

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

                # Calculate accuracy
                preds = (logits > 0).long()
                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

            avg_loss = total_loss / len(train_loader)
            accuracy = correct / total

        print(f'Probe trained with loss: {avg_loss :.3f}, accuracy: {accuracy :.3f}')


    def token_scores(self, X: torch.Tensor, use_sigmoid: bool = True) -> torch.Tensor:
        """
        X: [N_tokens, d_model]
        Returns:
            scores: [N_tokens]
            - if use_sigmoid=True, scores are in [0,1] (prob-like)
            - else, scores are raw logits
        """
        logits = self.forward(X)

        if use_sigmoid == True:
          return torch.sigmoid(logits)
        else:
          return logits

    @staticmethod
    def response_scores_from_tokens(token_scores, response_ids, agg='mean'):
        """
        Aggregate token-level scores to response-level scores.
        """
        response_scores = []

        unique_ids = response_ids.unique(sorted=True)



        for id in unique_ids:
            mask = (id == response_ids)
            resp_token_scores = token_scores[mask]

            if agg == 'mean':
                resp_score = resp_token_scores.mean()
            elif agg == 'max':
                resp_score = resp_token_scores.max()
            else:
                raise ValueError(f"agg must be 'mean' or 'max', got {agg}")

            response_scores.append(resp_score)

        response_scores = torch.stack(response_scores)

        return response_scores

    @staticmethod
    def auroc(response_labels: torch.Tensor, response_scores: torch.Tensor) -> float:
        """
        Compute AUROC for response-level scores vs labels.
        """

        labels_np = response_labels.cpu().numpy()
        scores_np = response_scores.cpu().numpy()


        labels_np = labels_np.astype(np.int32)


        scores_np = scores_np.astype(np.float32)


        labels_np = labels_np.ravel()
        scores_np = scores_np.ravel()



        from sklearn.metrics import roc_auc_score
        return roc_auc_score(labels_np, scores_np)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
probe = DeceptionProbe( lm.model.cfg.d_model, device=device)
probe.fit(ip_loader, num_epochs=45, lr = 0.001)

In [ ]:
@dataclass
class InsiderExample:
    text: str      # the assistant's report / reply we want to score
    label: int     # 0 = honest, 1 = deceptive

## Building the text dataset
* Loading the dataset from downloaded repo *(using os, json and dataclass)*
* Converting that to its text and span that we need to focus
* Building a full-fledged dataset from it - taking the text and labels, passing through the models, collecting activations, and then concatenating them together.

In [ ]:

def load_insider_trading_examples(path: str) -> List[InsiderExample]:
    """
    Load insider trading examples from a repo file
    and map them to (text, label).
    """
    examples: List[InsiderExample] = []

    # Open and read the JSON file
    with open(path, "r") as f:
        data = json.load(f)  # Parse entire JSON file into Python list

    # Iterate through each record
    for record in data:
        # Extract the assistant's final response from transcript
        transcript = record["transcript"]

        # Find last assistant message (the one we want to evaluate)
        assistant_messages = [msg for msg in transcript if msg["role"] == "assistant"]
        if not assistant_messages:
            continue  # Skip if no assistant message

        text = assistant_messages[-1]["content"]

        # Extract label from metadata
        made_trade = record["metadata"]["made_trade"]

        # Map to binary label: 0 = honest, 1 = deceptive
        if made_trade == "invalid":
            label = 1  # Deceptive (made insider trade)
        else:  # "yes" or "no"
            label = 0  # Honest (either legitimate trade or no trade)

        # Create InsiderExample and append
        examples.append(InsiderExample(text=text, label=label))

    return examples

In [ ]:
# Load the examples
examples = load_insider_trading_examples(
    "data/deception-detection/data/insider_trading/llama-70b-3.3-generations.json"
)

print(f"Loaded {len(examples)} examples")
print(f"\nFirst example:")
print(f"Label: {examples[0].label}")
print(f"Text preview: {examples[0].text[:200]}...")

# Check label distribution
num_deceptive = sum(1 for ex in examples if ex.label == 1)
num_honest = sum(1 for ex in examples if ex.label == 0)
print(f"\nLabel distribution:")
print(f"  Honest (0): {num_honest}")
print(f"  Deceptive (1): {num_deceptive}")

In [ ]:

def build_insider_text_and_span(ex: InsiderExample) -> Tuple[str, Tuple[int, int]]:
    """
    Construct a text string to feed into the model, plus
    the span (start_idx, end_idx) of the assistant tokens.
    """
    # TODO (simple version):
    #   For now, just set full_text = ex.text (no extra system/user context).
    #   Then treat the entire sequence as the span: (0, None) or (0, seq_len).
    #
    # Later:
    #   If you include system+user messages here, you can make the span
    #   start at the assistant segment only.

    full_text = ex.text
    span = (0,None)

    return full_text, span


In [ ]:
def build_eval_token_dataset(
    lm: LMWithHooks,
    examples: List[InsiderExample],
    cfg: Config,
    layer,
    K: int = 16
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Build token-level dataset for evaluation.

    Returns:
        X_tokens: [N_tokens, d_model]
        response_ids: [N_tokens]  (0..N_responses-1)
        y_responses: [N_responses]
    """
    X_list = []
    resp_id_list = []
    y_resp_list = []

    for resp_id, ex in enumerate(tqdm(examples, desc="Building eval tokens")):
        full_text, span = build_insider_text_and_span(ex)

        resid = lm.get_resid_for_text(full_text, layer)  # [seq, d_model]
        resid_slice = resid[-K:, :]

        X_list.append(resid_slice)
        y_resp_list.append(ex.label)
        resp_id_list.append(torch.full((resid_slice.shape[0],), resp_id, dtype=torch.long))

    X_tokens = torch.cat(X_list, dim = 0)
    y_responses = torch.tensor(y_resp_list, dtype = torch.long)
    response_ids = torch.cat(resp_id_list, dim =0)

    return X_tokens, response_ids, y_responses


##Evaluate Probe on Insider Trading Dataset

In [ ]:
def evaluate_probe_on_insider(
    lm: LMWithHooks,
    probe: DeceptionProbe,
    examples: List[InsiderExample],
    cfg: Config,
    layer,
    K: int = 16
):
    """
    Evaluate probe on insider trading dataset.

    Returns:
        AUROC score
    """
    # Build token-level dataset
    X_tokens, response_ids, y_responses = build_eval_token_dataset(lm, examples, cfg, layer,K=K)

    # Move tokens to device for probe forward pass
    X_tokens = X_tokens.to(cfg.device)

    with torch.no_grad():
        # Get token-level logits
        token_logits = probe(X_tokens)

        # Convert logits to scores (probabilities)
        token_scores = torch.sigmoid(token_logits)

        # Move to CPU for aggregation (both tensors must be on same device!)
        token_scores_cpu = token_scores.cpu()
        response_ids_cpu = response_ids.cpu()
        y_responses_cpu = y_responses.cpu()

        # Aggregate token scores to response scores
        response_scores = probe.response_scores_from_tokens(
            token_scores_cpu,
            response_ids_cpu,
            agg='mean'  # or 'max'
        )

        # Compute AUROC

        auc = probe.auroc(y_responses_cpu, response_scores)

    return auc




In [ ]:

examples = load_insider_trading_examples("data/deception-detection/data/insider_trading/llama-70b-3.3-generations.json")



In [ ]:
print(f'AUROC for Insider dataset: {evaluate_probe_on_insider(lm, probe, examples = examples[:10], cfg = cfg, layer = cfg.probe_layer)}')

**Ther AUC performance for GPT2 small was not that great, this is probably because its quite powerless in learning the deception behavior.**

So, now I want to sweep over middle-end layers in **Qwen 2.5-0.5B**model, train probes on them, get evaluation scores for them.

- First, the IP dataset for each layer has to be created - these are the activations from that layer. This function takes in cfg, and trains on that layer using cfg.probe_layer - need to change it. So instead of cfg, we need to pass in layer, and we have to make changes downstream.

- The same has to be done for the evaluate_probe_on_insider function and others.

- Below is a function that sweeps over some of the layers in Qwen 2.5-0.5B model.

##Sweep over Middle layers of Qwen 2.5-0.5B

In [ ]:
def sweep_over_layers_for_auc(layer_list =[16,17,18,19,20,21], lm = lm, facts = facts, cfg = cfg,):

  auc_scores = {}

  for layer in layer_list:

    X_ip, y_ip = build_ip_token_dataset(lm, facts, cfg, layer)
    ip_loader = make_ip_dataloader(X_ip, y_ip, cfg)


    probe = DeceptionProbe( lm.model.cfg.d_model, device=device)

    probe.fit(ip_loader, num_epochs=45, lr = 0.001)

    probe_auc = evaluate_probe_on_insider(lm, probe, examples = examples, cfg = cfg, layer = cfg.probe_layer)
    auc_scores[layer] = probe_auc
    print('\n')
    print(f'AUC for layer {layer}: {probe_auc :.3f}')
    print('\n')

  return auc_scores

In [ ]:
auc_scores_layers = sweep_over_layers_for_auc()